In [ ]:
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieRecommendationSystem") \
    .getOrCreate()

# Load CSV files into PySpark DataFrames
ratings = spark.read.csv('ml-latest-small/ratings.csv', header=True, inferSchema=True)
movies = spark.read.csv('ml-latest-small/movies.csv', header=True, inferSchema=True)

# Inspect ratings structure
ratings.select('userId', 'movieId', 'rating').show(5)

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      1|   4.0|
|     1|      3|   4.0|
|     1|      6|   4.0|
|     1|     47|   5.0|
|     1|     50|   5.0|
+------+-------+------+
only showing top 5 rows


In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Split dataset into 80% training and 20% testing
train_data, test_data = ratings.randomSplit([0.8, 0.2], seed=42)

# Build ALS recommendation model
als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop"
)

model = als.fit(train_data)

# Evaluate model performance using Root Mean Squared Error (RMSE)
predictions = model.transform(test_data)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) = {rmse:.4f}")

Root Mean Squared Error (RMSE) = 0.8767


In [ ]:
# Generate Top 5 movie recommendations for all users
user_recommendations = model.recommendForAllUsers(5)

# Example: Get top 5 recommendations for User ID 10
user_id = 10
user_recs = user_recommendations.filter(f"userId = {user_id}").collect()

if user_recs:
    rec_movie_ids = [row.movieId for row in user_recs[0].recommendations]

    # Display the actual movie titles instead of raw IDs
    recommended_movies = movies.filter(movies.movieId.isin(rec_movie_ids))
    print(f"Top 5 Movie Recommendations for User {user_id}:")
    recommended_movies.select("title", "genres").show(truncate=False)

Top 5 Movie Recommendations for User 10:
+-------------------------------+-----------------------------+
|title                          |genres                       |
+-------------------------------+-----------------------------+
|Life Less Ordinary, A (1997)   |Romance|Thriller             |
|Practical Magic (1998)         |Drama|Fantasy|Mystery|Romance|
|Rules of Attraction, The (2002)|Comedy|Drama|Romance|Thriller|
|CB4 - The Movie (1993)         |Comedy                       |
|First Daughter (2004)          |Comedy|Romance               |
+-------------------------------+-----------------------------+



In [ ]:
print("\n--- Model Explanation ---")
print("The model used is Alternating Least Squares (ALS). ALS is a collaborative filtering algorithm commonly used for recommendation systems.")
print("It works by factorizing the user-item interaction matrix into two lower-rank matrices: one for users and one for items. These latent factors are then used to predict missing ratings or recommend items.")
print("ALS is particularly suitable for implicit feedback datasets and can handle large sparse matrices efficiently.")

print("\n--- Additional Model Evaluation Statistics ---")
# Mean Squared Error (MSE)
evaluator_mse = RegressionEvaluator(metricName="mse", labelCol="rating", predictionCol="prediction")
mse = evaluator_mse.evaluate(predictions)
print(f"Mean Squared Error (MSE) = {mse:.4f}")

# Mean Absolute Error (MAE)
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")
mae = evaluator_mae.evaluate(predictions)
print(f"Mean Absolute Error (MAE) = {mae:.4f}")

# R-squared (R2)
evaluator_r2 = RegressionEvaluator(metricName="r2", labelCol="rating", predictionCol="prediction")
r2 = evaluator_r2.evaluate(predictions)
print(f"R-squared (R2) = {r2:.4f}")

# The RMSE was already calculated, but we can print it again for completeness
print(f"Root Mean Squared Error (RMSE) = {rmse:.4f}")


--- Model Explanation ---
The model used is Alternating Least Squares (ALS). ALS is a collaborative filtering algorithm commonly used for recommendation systems.
It works by factorizing the user-item interaction matrix into two lower-rank matrices: one for users and one for items. These latent factors are then used to predict missing ratings or recommend items.
ALS is particularly suitable for implicit feedback datasets and can handle large sparse matrices efficiently.

--- Additional Model Evaluation Statistics ---
Mean Squared Error (MSE) = 0.7686
Mean Absolute Error (MAE) = 0.6769
R-squared (R2) = 0.2962
Root Mean Squared Error (RMSE) = 0.8767


In [ ]:
print("\n--- Attempting to improve model accuracy through hyperparameter tuning ---")

# Define new hyperparameters
# Increasing maxIter can help the model converge better, but also increases computation time.
# Adjusting regParam can help prevent overfitting.
new_max_iter = 20  # Increased from 10
new_reg_param = 0.15 # Slightly adjusted from 0.1

print(f"Retraining ALS model with maxIter={new_max_iter} and regParam={new_reg_param}")

# Build a new ALS recommendation model with updated hyperparameters
als_tuned = ALS(
    maxIter=new_max_iter,
    regParam=new_reg_param,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42 # Ensure reproducibility
)

# Train the new model
model_tuned = als_tuned.fit(train_data)

# Evaluate the tuned model performance
predictions_tuned = model_tuned.transform(test_data)
evaluator_tuned = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse_tuned = evaluator_tuned.evaluate(predictions_tuned)

print(f"Root Mean Squared Error (RMSE) with tuned parameters = {rmse_tuned:.4f}")
print(f"Previous RMSE = {rmse:.4f}")

if rmse_tuned < rmse:
    print("Accuracy improved! The new RMSE is lower.")
else:
    print("Accuracy did not improve significantly or worsened with these parameters. Further tuning might be required.")


--- Attempting to improve model accuracy through hyperparameter tuning ---
Retraining ALS model with maxIter=20 and regParam=0.15
Root Mean Squared Error (RMSE) with tuned parameters = 0.8679
Previous RMSE = 0.8767
Accuracy improved! The new RMSE is lower.


In [ ]:
print("\n--- Attempting more aggressive hyperparameter tuning with a higher rank ---")

# Define new hyperparameters, specifically increasing 'rank'
# 'rank' is the number of latent factors in the model (e.g., how many characteristics each user/movie has)
new_max_iter_higher_rank = 20
new_reg_param_higher_rank = 0.15
new_rank = 20  # Increased from default (which is usually around 10 or less internally if not specified)

print(f"Retraining ALS model with maxIter={new_max_iter_higher_rank}, regParam={new_reg_param_higher_rank}, and rank={new_rank}")

# Build a new ALS recommendation model with updated hyperparameters including rank
als_higher_rank = ALS(
    maxIter=new_max_iter_higher_rank,
    regParam=new_reg_param_higher_rank,
    rank=new_rank,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42 # Ensure reproducibility
)

# Train the new model
model_higher_rank = als_higher_rank.fit(train_data)

# Evaluate the tuned model performance
predictions_higher_rank = model_higher_rank.transform(test_data)
evaluator_higher_rank = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse_higher_rank = evaluator_higher_rank.evaluate(predictions_higher_rank)

print(f"Root Mean Squared Error (RMSE) with higher rank parameters = {rmse_higher_rank:.4f}")
print(f"Previous Tuned RMSE = {rmse_tuned:.4f}")

if rmse_higher_rank < rmse_tuned:
    print("Accuracy improved further! The new RMSE is lower.")
else:
    print("Accuracy did not improve or worsened. Further tuning or a different model might be needed.")

# You could also explore more advanced models like implementing SVD from scratch or using external libraries
# if PySpark's ALS does not meet desired accuracy, but for typical use-cases, tuning ALS is often sufficient.



--- Attempting more aggressive hyperparameter tuning with a higher rank ---
Retraining ALS model with maxIter=20, regParam=0.15, and rank=20
Root Mean Squared Error (RMSE) with higher rank parameters = 0.8683
Previous Tuned RMSE = 0.8679
Accuracy did not improve or worsened. Further tuning or a different model might be needed.


In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

print("\n--- Performing systematic hyperparameter tuning for ALS using Cross-Validation ---")

# Define the ALS model
als_cv = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
)

# Define a parameter grid to search over
# We wrap this in parentheses to allow multi-line chaining without backslashes
param_grid = (
    ParamGridBuilder()
    .addGrid(als_cv.rank, [10, 20, 30])
    .addGrid(als_cv.regParam, [0.01, 0.1, 0.2])
    .addGrid(als_cv.maxIter, [10, 15, 20])
    .build()
)

# Define the evaluator
evaluator_cv = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

# Set up the CrossValidator
cv = CrossValidator(
    estimator=als_cv,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_cv,
    numFolds=3,
    seed=42
)

# Run cross-validation to find the best model
print("Running Cross-Validation... This might take a while.")
cv_model = cv.fit(train_data)

# Get the best model from cross-validation
best_als_model = cv_model.bestModel

print("\n--- Best ALS Model Hyperparameters ---")
print(f"Rank: {best_als_model._java_obj.parent().getRank()}")
print(f"Regularization Parameter: {best_als_model._java_obj.parent().getRegParam()}")
print(f"Max Iterations: {best_als_model._java_obj.parent().getMaxIter()}")

# Evaluate the best model on the test data
predictions_best_als = best_als_model.transform(test_data)
rmse_best_als = evaluator_cv.evaluate(predictions_best_als)

print(f"\nRoot Mean Squared Error (RMSE) with best ALS parameters = {rmse_best_als:.4f}")
print(f"Previous best manual RMSE = {rmse_tuned:.4f}")

if rmse_best_als < rmse_tuned:
    print("Significant accuracy improvement! The new RMSE is lower with optimized hyperparameters.")
else:
    print("Accuracy did not significantly improve. We can discuss deep learning alternatives next.")


--- Performing systematic hyperparameter tuning for ALS using Cross-Validation ---
Running Cross-Validation... This might take a while.

--- Best ALS Model Hyperparameters ---
Rank: 10
Regularization Parameter: 0.2
Max Iterations: 20

Root Mean Squared Error (RMSE) with best ALS parameters = 0.8751
Previous best manual RMSE = 0.8679
Accuracy did not significantly improve. We can discuss deep learning alternatives next.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np

print("--- Building a Deep Learning Model (NCF) with GPU Support ---")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Prepare Data
pdf_train = train_data.select("userId", "movieId", "rating").toPandas()
pdf_test = test_data.select("userId", "movieId", "rating").toPandas()

user_to_index = {u: i for i, u in enumerate(pdf_train['userId'].unique())}
movie_to_index = {m: i for i, m in enumerate(pdf_train['movieId'].unique())}

num_users = len(user_to_index)
num_movies = len(movie_to_index)

class MovieDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor([user_to_index.get(u, 0) for u in df['userId']], dtype=torch.long)
        self.movies = torch.tensor([movie_to_index.get(m, 0) for m in df['movieId']], dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

train_loader = DataLoader(MovieDataset(pdf_train), batch_size=256, shuffle=True)
test_loader = DataLoader(MovieDataset(pdf_test), batch_size=256)

# 2. Define Model
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_size=64):
        super(NCF, self).__init__()
        self.user_embed = nn.Embedding(num_users + 1, embedding_size)
        self.item_embed = nn.Embedding(num_items + 1, embedding_size)
        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, item):
        user_e = self.user_embed(user)
        item_e = self.item_embed(item)
        x = torch.cat([user_e, item_e], dim=-1)
        return self.fc_layers(x).squeeze()

model_dl = NCF(num_users, num_movies).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_dl.parameters(), lr=0.001)

# 3. Training Loop (Increased epochs for GPU)
model_dl.train()
epochs = 10
print(f"Training for {epochs} epochs...")
for epoch in range(epochs):
    total_loss = 0
    for u, m, r in train_loader:
        u, m, r = u.to(device), m.to(device), r.to(device)
        optimizer.zero_grad()
        outputs = model_dl(u, m)
        loss = criterion(outputs, r)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# 4. Evaluation
model_dl.eval()
all_preds = []
all_ratings = []
with torch.no_grad():
    for u, m, r in test_loader:
        u, m = u.to(device), m.to(device)
        preds = model_dl(u, m)
        all_preds.extend(preds.cpu().numpy())
        all_ratings.extend(r.numpy())

rmse_dl = np.sqrt(np.mean((np.array(all_preds) - np.array(all_ratings))**2))
print(f"\nDeep Learning (NCF) RMSE = {rmse_dl:.4f}")
print(f"Target ALS RMSE to beat = 0.8679")

--- Building a Deep Learning Model (NCF) with GPU Support ---
Using device: cuda
Training for 10 epochs...
Epoch 1/10, Loss: 1.5250
Epoch 2/10, Loss: 0.9350
Epoch 3/10, Loss: 0.8695
Epoch 4/10, Loss: 0.8331
Epoch 5/10, Loss: 0.7969
Epoch 6/10, Loss: 0.7718
Epoch 7/10, Loss: 0.7456
Epoch 8/10, Loss: 0.7293
Epoch 9/10, Loss: 0.7131
Epoch 10/10, Loss: 0.6960

Deep Learning (NCF) RMSE = 0.8944
Target ALS RMSE to beat = 0.8679
